tab_plaqerr — Table `tab:plaqerr` (plaquette: time-averaged relative error of N(t))
Cells: mean over the five loss rates of err_N = mean_t |N - N_exact| / N(0), 0 <= tJ <= 200; last column: mean over
all trajectories of a scheme. Diverged runs are excluded (footnotes). Writes `tab_plaqerr.tex` and prints the table.

In [1]:
USE_OUTPUT = False
OUTPUT_DIR = "output"

In [2]:
import sys
from pathlib import Path
HERE = Path.cwd()
sys.path.insert(0, str(HERE.parent))
from common import io as IO
if USE_OUTPUT:
    import postprocess
    D = postprocess.build(OUTPUT_DIR)
else:
    D = IO.read_data(HERE / "data")
T, RUNS = D["table.csv"], D["runs.csv"]

In [3]:
ICS = ["slater", "cdw", "mixed", "one_pair", "polarized", "3u1d", "neel", "mean"]
HEAD = ["Slater", "CDW", "mixed", "pair", "polarized", r"$3\!\uparrow\!1\!\downarrow$", r"N\'eel", "mean"]
NAME = {"HF": "MF", "2B": "2B", "TPPX": "TPPX", "mGW": r"m$GW$", "DSL": "DSL"}
MARK = {}                                     # footnote marks for cells with excluded (diverged) trajectories
foot = []
for m, sym in (("mGW", "*"), ("DSL", r"\dagger")):
    for ic in ICS[:-1]:
        c = IO.select(T, method=m, ic=ic)
        if c["n_excluded"][0]:
            MARK[(m, ic)] = sym
            bad = IO.select(IO.select(RUNS, method=m, ic=ic), status="Diverged")
            foot.append(f"{sym}: excludes {m} {ic} at gamma/J = {', '.join(f'{g:g}' for g in bad['gamma'])} (Diverged)")
lines = [" & " + " & ".join(HEAD) + r" \\", r"\hline"]
for m in ["HF", "2B", "TPPX", "mGW", "DSL"]:
    cells = []
    for ic in ICS:
        v = IO.select(T, method=m, ic=ic)["err_N"][0]
        s = f"{v:#.2g}"
        cells.append(s + (f"$^{MARK[(m, ic)]}$" if (m, ic) in MARK else ""))
    lines.append(f"{NAME[m]:6s}& " + " & ".join(cells) + r" \\")
tex = "\\begin{tabular}{lcccccccc}\n" + "\n".join(lines) + "\n\\end{tabular}\n"
(HERE / "tab_plaqerr.tex").write_text(tex)
print(tex)
print("\n".join(foot))

\begin{tabular}{lcccccccc}
 & Slater & CDW & mixed & pair & polarized & $3\!\uparrow\!1\!\downarrow$ & N\'eel & mean \\
\hline
MF    & 0.033 & 0.018 & 0.34 & 0.032 & 0.060 & 0.0011 & 0.29 & 0.11 \\
2B    & 0.34 & 0.37 & 0.029 & 0.73 & 0.056 & 0.00092 & 0.069 & 0.23 \\
TPPX  & 0.044 & 0.052 & 0.28 & 0.074 & 0.058 & 0.00074 & 0.33 & 0.12 \\
m$GW$ & 0.77 & 0.85 & 0.57 & 1.7 & 0.38 & 0.28$^*$ & 0.48 & 0.73 \\
DSL   & 0.016 & 0.0074 & 0.014 & 0.0047 & 0.0092 & 0.00043 & 0.0081$^\dagger$ & 0.0085 \\
\end{tabular}

*: excludes mGW 3u1d at gamma/J = 1 (Diverged)
\dagger: excludes DSL neel at gamma/J = 10 (Diverged)
